# LeetCode 654 — Maximum Binary Tree bằng Python

Link đề: https://leetcode.com/problems/maximum-binary-tree/ · Độ khó: **Medium** · Chủ đề: Stack, Tree, Array, Divide and Conquer, Binary Tree, Cartesian Tree, Monotonic Stack

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        ...
```

Ghi chú: LeetCode trả về **nút gốc** của cây, và hiển thị cây dưới dạng list kiểu `[6,3,5,null,2,0,null,null,1]`. Notebook tự dựng `TreeNode`, hàm `build_tree` (list → cây) và `tree_to_list` (cây → list kiểu LeetCode) để chạy và so sánh. Các ví dụ số 3, 4 và các test tự đặt đã được kiểm chứng bằng code; nếu cần hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ mảng `nums` **không trùng giá trị**, dựng cây theo quy tắc "gốc là phần tử lớn nhất; nửa trái dựng cây con trái; nửa phải dựng cây con phải", rồi trả về gốc.
2. Nắm các công cụ Python: định nghĩa `TreeNode`, dựng và xuất cây dạng list kiểu LeetCode, cắt lát (`nums[:i]`, `nums[i+1:]`), `max` + `index`, đệ quy trên đoạn chỉ số `[l, r]`, danh sách dùng như **ngăn xếp** (`append`, `pop`, `[-1]`), giới hạn độ sâu đệ quy.
3. Đi từ **đệ quy "tìm max rồi cắt đôi"** (`O(n²)` xấu nhất) → **bảng thưa Sparse Table** để tìm max trong `O(1)` (`O(n log n)`) → **ngăn xếp đơn điệu** dựng cây trong **một lượt** (`O(n)`) → **luật cha = "lớn hơn gần nhất" nhỏ hơn trong hai phía** (`O(n)`).
4. Hiểu mấu chốt: đây là **cây Descartes (Cartesian tree)** — duyệt **giữa** (in-order) cây trả về đúng mảng ban đầu, và mỗi nút **lớn hơn** các con của nó. Hai tính chất này xác định duy nhất cây khi các giá trị khác nhau.
5. Tránh các lỗi kinh điển: quên cộng độ lệch chỉ số khi đệ quy theo đoạn, cắt lát sai làm đệ quy vô hạn, quên gắn lại cây con bị pop, trả về `stack[-1]` thay vì `stack[0]`, và không xử lý mảng rỗng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` **không có phần tử trùng nhau**. Một **cây nhị phân lớn nhất** (maximum binary tree) được dựng đệ quy từ `nums` như sau:

1. Tạo nút gốc có giá trị bằng **phần tử lớn nhất** của `nums`.
2. Dựng **cây con trái** đệ quy từ **đoạn đầu** nằm **bên trái** phần tử lớn nhất.
3. Dựng **cây con phải** đệ quy từ **đoạn đuôi** nằm **bên phải** phần tử lớn nhất.

Trả về **cây nhị phân lớn nhất** dựng từ `nums`.

**Ví dụ 1**

- Input: `nums = [3,2,1,6,0,5]`
- Output: `[6,3,5,null,2,0,null,null,1]`
- Giải thích: phần tử lớn nhất là `6`; bên trái `[3,2,1]`, bên phải `[0,5]`. Trong `[3,2,1]` lớn nhất là `3` (không có bên trái, bên phải `[2,1]`) và cứ thế. Trong `[0,5]` lớn nhất là `5`, bên trái là `[0]`, không có bên phải.

**Ví dụ 2**

- Input: `nums = [3,2,1]`
- Output: `[3,null,2,null,1]`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [1]`
- Output: `[1]`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [1,2,3]`
- Output: `[3,2,null,1]` (cây lệch trái)

**Ràng buộc:**

- `1 <= nums.length <= 1000`
- `0 <= nums[i] <= 1000`
- Mọi số trong `nums` **khác nhau từng đôi một**.

Chú ý: cây trả về chỉ có **một** đáp án đúng (vì các giá trị khác nhau, phần tử lớn nhất luôn rõ ràng). Có thể có cây lệch sâu `1000` tầng (`nums` giảm dần hoặc tăng dần), nên đệ quy Python phải được nới giới hạn độ sâu.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


sys.setrecursionlimit(10000)      # cây lệch có thể sâu 1000 tầng

examples = [
    ([3, 2, 1, 6, 0, 5], [6, 3, 5, None, 2, 0, None, None, 1]),
    ([3, 2, 1], [3, None, 2, None, 1]),
    ([1], [1]),                         # ví dụ tự đặt
    ([1, 2, 3], [3, 2, None, 1]),       # ví dụ tự đặt
]
for nums, expected in examples:
    assert tree_to_list(build_tree(expected)) == expected      # kiểm tra hai hàm dựng/xuất
    print(nums, "-> kỳ vọng", expected)

[3, 2, 1, 6, 0, 5] -> kỳ vọng [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> kỳ vọng [3, None, 2, None, 1]
[1] -> kỳ vọng [1]
[1, 2, 3] -> kỳ vọng [3, 2, None, 1]


## 3. Kiến thức nền

### 3.1. Nút cây và cây nhị phân

Mỗi nút (`TreeNode`) có `val` (giá trị), `left`, `right` (nút con hoặc `None`). Chỉ cần giữ **nút gốc** là truy cập được cả cây. LeetCode định nghĩa sẵn `TreeNode`; ở đây ta tự định nghĩa ở cell trên để chạy được tại máy.

In [2]:
leaf_a = TreeNode(2)
leaf_b = TreeNode(5)
root = TreeNode(6, leaf_a, leaf_b)
print(root.val, root.left.val, root.right.val)
print("con trái của lá:", leaf_a.left)        # None: lá không có con

6 2 5
con trái của lá: None


### 3.2. Dựng và xuất cây kiểu LeetCode

Đề viết cây dạng list theo **thứ tự từng tầng, từ trái sang phải**, `None` (hiện là `null`) là chỗ trống. `build_tree` dựng cây từ list (dùng hàng đợi `deque`), `tree_to_list` làm ngược lại và **cắt các `None` thừa ở cuối**. Với bài này, đáp án cần xuất ra đúng dạng list để so sánh với đề. `show_levels` in từng tầng cho dễ nhìn:

In [3]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"tầng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

t = build_tree([6, 3, 5, None, 2, 0, None, None, 1])
show_levels(t)
print(tree_to_list(t))

tầng 0: [6]
tầng 1: [3, 5]
tầng 2: [None, 2, 0, None]
tầng 3: [None, 1, None, None]
[6, 3, 5, None, 2, 0, None, None, 1]


### 3.3. Tìm phần tử lớn nhất và vị trí của nó; cắt lát

- `max(nums)` cho giá trị lớn nhất; `nums.index(x)` cho **vị trí** (chỉ số) đầu tiên của `x`.
- `nums[:i]` là đoạn **trước** vị trí `i` (không gồm `i`); `nums[i+1:]` là đoạn **sau** vị trí `i`.
- Mỗi lần cắt lát tạo **bản sao** list mới (tốn thời gian và bộ nhớ tỉ lệ độ dài).

In [4]:
nums = [3, 2, 1, 6, 0, 5]
m = max(nums)
i = nums.index(m)
print("max =", m, "ở vị trí", i)
print("trái :", nums[:i])        # [3, 2, 1]
print("phải :", nums[i + 1:])    # [0, 5]
print("rỗng :", nums[6:], nums[:0])    # đoạn rỗng là list rỗng, không lỗi

max = 6 ở vị trí 3
trái : [3, 2, 1]
phải : [0, 5]
rỗng : [] []


### 3.4. Đệ quy trên đoạn chỉ số `[l, r]` thay vì cắt lát

Thay vì cắt `nums` thành list mới, ta có thể giữ **nguyên** `nums` và truyền hai chỉ số `l`, `r` (đoạn đóng `nums[l..r]`). Đoạn rỗng khi `l > r`. Tìm vị trí lớn nhất trong đoạn: `max(range(l, r + 1), key=nums.__getitem__)` hoặc vòng `for`. Cách này không sao chép dữ liệu. Chú ý: vị trí tìm được **đã là chỉ số toàn cục**, không cần cộng thêm `l` (nhưng nếu ta tìm trên lát cắt `nums[l:r+1]` thì phải cộng `l` — nguồn lỗi kinh điển, xem mục 13).

In [5]:
nums = [3, 2, 1, 6, 0, 5]
l, r = 4, 5                                   # đoạn [0, 5]
pos = max(range(l, r + 1), key=lambda k: nums[k])
print("vị trí toàn cục của max trong nums[4..5]:", pos, "giá trị", nums[pos])
# cách sai quen thuộc: tìm trên lát cắt rồi quên cộng l
local = nums[l:r + 1].index(max(nums[l:r + 1]))
print("vị trí trong lát cắt:", local, "-> toàn cục phải là", local + l)

vị trí toàn cục của max trong nums[4..5]: 5 giá trị 5
vị trí trong lát cắt: 1 -> toàn cục phải là 5


### 3.5. List dùng như ngăn xếp (stack)

**Ngăn xếp** là cấu trúc "vào sau, ra trước". Với list Python: `stack.append(x)` đẩy vào đỉnh, `stack.pop()` lấy và bỏ đỉnh, `stack[-1]` xem đỉnh (không bỏ). Kiểm tra rỗng bằng `if stack:`. Dùng một ngăn xếp mà **giá trị luôn giảm dần từ đáy lên đỉnh** gọi là **ngăn xếp đơn điệu giảm** — ta sẽ dùng nó để dựng cây trong một lượt.

In [6]:
stack = []
for x in [5, 3, 4, 1]:
    while stack and stack[-1] < x:        # bỏ những phần tử nhỏ hơn x
        print("  pop", stack.pop())
    stack.append(x)
    print("đẩy", x, "->", stack)          # luôn giảm dần từ đáy lên đỉnh

đẩy 5 -> [5]
đẩy 3 -> [5, 3]
  pop 3
đẩy 4 -> [5, 4]
đẩy 1 -> [5, 4, 1]


### 3.5b. Cây Descartes và hai tính chất nhận dạng

Cây trong đề còn gọi là **Cartesian tree** (cây Descartes) của mảng. Với các giá trị khác nhau, nó là **cây duy nhất** thoả cả hai tính chất:

1. **Duyệt giữa (in-order)** (trái, gốc, phải) trả về **đúng thứ tự mảng ban đầu**.
2. **Tính chất đống (heap, max-heap)**: giá trị mỗi nút **lớn hơn** giá trị của cả hai con.

Ta dùng hai tính chất này làm **hàm kiểm tra tính hợp lệ** độc lập (không dựa vào cách dựng) trong phần test.

In [7]:
def inorder(root):
    out = []
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        out.append(node.val)
        node = node.right
    return out

def heap_ok(root):
    if root is None:
        return True
    for ch in (root.left, root.right):
        if ch is not None and ch.val >= root.val:
            return False
    return heap_ok(root.left) and heap_ok(root.right)

def is_valid_max_tree(root, nums):
    return inorder(root) == list(nums) and heap_ok(root)

t = build_tree([6, 3, 5, None, 2, 0, None, None, 1])
print(inorder(t), heap_ok(t), is_valid_max_tree(t, [3, 2, 1, 6, 0, 5]))

[3, 2, 1, 6, 0, 5] True True


### 3.6. Giới hạn độ sâu đệ quy

Python mặc định chỉ cho đệ quy sâu khoảng `1000` tầng (`sys.getrecursionlimit()`). Mảng **tăng dần hoặc giảm dần** `1000` phần tử cho cây lệch sâu `1000`, cộng thêm vài tầng gọi hàm bao ngoài sẽ **vượt giới hạn mặc định**. LeetCode thường nới giới hạn nên code đệ quy vẫn qua, nhưng ở máy cá nhân ta gọi `sys.setrecursionlimit(10000)` (đã làm ở cell đầu). Các cách dùng ngăn xếp (cách 3, 4) **không đệ quy** nên không có vấn đề này.

In [8]:
print("giới hạn hiện tại:", sys.getrecursionlimit())

def depth_demo(n):
    return 0 if n == 0 else 1 + depth_demo(n - 1)

print(depth_demo(1500))      # chạy được vì đã nới giới hạn

giới hạn hiện tại: 10000
1500


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Dựng tay cây từ `nums = [3, 2, 1, 6, 0, 5]`:

1. Lớn nhất là `6` (vị trí 3) → gốc `6`. Trái `[3, 2, 1]`, phải `[0, 5]`.
2. Với `[3, 2, 1]`: lớn nhất `3` → trái rỗng, phải `[2, 1]`. Với `[2, 1]`: lớn nhất `2` → trái rỗng, phải `[1]`. Với `[1]`: nút `1`.
3. Với `[0, 5]`: lớn nhất `5` → trái `[0]` → nút `0`, phải rỗng.

```
        6
      /   \
     3     5
      \   /
       2 0
        \
         1
```

Đọc ra list từng tầng: `[6, 3, 5, null, 2, 0, null, null, 1]` — đúng đáp án.

Nhận xét: **mỗi phần tử trở thành một nút**; phần tử nào lớn hơn sẽ nằm **cao hơn** (gần gốc hơn). Mỗi lần tìm max trong một đoạn tốn thời gian tuyến tính theo độ dài đoạn.

In [9]:
# Kiểm tra bằng tay-code: dựng cây từ list kỳ vọng và đối chiếu với tính chất
t = build_tree([6, 3, 5, None, 2, 0, None, None, 1])
show_levels(t)
print("duyệt giữa:", inorder(t), "= nums ban đầu [3, 2, 1, 6, 0, 5]")
print("heap ok   :", heap_ok(t))

tầng 0: [6]
tầng 1: [3, 5]
tầng 2: [None, 2, 0, None]
tầng 3: [None, 1, None, None]
duyệt giữa: [3, 2, 1, 6, 0, 5] = nums ban đầu [3, 2, 1, 6, 0, 5]
heap ok   : True


## 5. Cách 1 — Brute Force / baseline: đệ quy "tìm max rồi cắt đôi"

**Ý tưởng "dịch nguyên văn đề bài":**

1. Nếu `nums` rỗng → trả `None`.
2. Tìm `m = max(nums)` và vị trí `i = nums.index(m)`.
3. `root = TreeNode(m)`, `root.left` = dựng từ `nums[:i]`, `root.right` = dựng từ `nums[i+1:]`.

Đây là cách trực tiếp nhất và ta dùng nó làm **oracle** (đáp án đối chứng độc lập) cho các cách nhanh hơn.

In [10]:
def build_slice(nums):
    if not nums:
        return None
    m = max(nums)
    i = nums.index(m)
    return TreeNode(m, build_slice(nums[:i]), build_slice(nums[i + 1:]))

for nums, expected in examples:
    got = tree_to_list(build_slice(nums))
    print(nums, "->", got)
    assert got == expected

[3, 2, 1, 6, 0, 5] -> [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> [3, None, 2, None, 1]
[1] -> [1]
[1, 2, 3] -> [3, 2, None, 1]


**Độ phức tạp.** Mỗi lời gọi quét cả đoạn để tìm max (`max`, `index`) và cắt lát hai lần.

- **Time:** `O(n²)` trường hợp xấu — mảng tăng/giảm dần: các đoạn có độ dài `n, n−1, ..., 1` → `~ n²/2` thao tác. Trường hợp cân bằng `O(n log n)`. Với `n ≤ 1000`, `n²/2 = 5·10^5` nên vẫn rất nhanh.
- **Space:** `O(n)` cho cây, cộng cắt lát và ngăn xếp đệ quy: `O(n²)` bộ nhớ tạm cộng dồn ở trường hợp xấu (nhưng mỗi bản sao được giải phóng khi hàm kết thúc).

**Nút thắt:** việc **tìm lại max** trong các đoạn con lặp đi lặp lại. Có thể tìm vị trí max của một đoạn bất kỳ trong `O(1)` nếu chuẩn bị trước; hoặc tránh hẳn việc tìm max bằng cách nhìn cấu trúc cây.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng cải tiến, cùng dựa vào tính chất Cartesian tree:

**Hướng A — Tìm max nhanh hơn.** Ta lặp lại câu hỏi "vị trí phần tử lớn nhất trong `nums[l..r]`" rất nhiều lần. Chuẩn bị **bảng thưa (sparse table)** một lần: `table[k][i]` = vị trí phần tử lớn nhất trong đoạn dài `2^k` bắt đầu tại `i`. Truy vấn bất kỳ đoạn `[l, r]` chỉ cần so hai ô chồng nhau → `O(1)`. Khi đó mỗi nút của cây tốn `O(1)`, cả cây `O(n)` cộng `O(n log n)` chuẩn bị.

**Hướng B — Dựng cây trong một lượt từ trái sang phải.** Xét lúc ta thêm `nums[i]` vào cây đang dựng từ `nums[0..i-1]`. Nút mới `x` phải nằm ở **"gai phải"** của cây hiện có (dãy các nút từ gốc đi mãi sang phải), và các nút trên gai phải có giá trị **giảm dần** từ gốc xuống. `x` sẽ chen vào chỗ đầu tiên trên gai mà nút ở đó **nhỏ hơn** `x`: phần gai từ đó trở xuống (các nút nhỏ hơn `x`) trở thành **cây con trái** của `x`, còn `x` thành con phải của nút lớn hơn nó ngay phía trên. Gai phải chính là một **ngăn xếp đơn điệu giảm**, và mỗi nút bị pop **tối đa một lần** → tổng `O(n)`.

Hướng B còn cho một cách nhìn thứ hai: **cha của `nums[i]`** là phần tử **nhỏ hơn** trong hai phần tử "lớn hơn gần nhất bên trái" và "lớn hơn gần nhất bên phải" (cách 4).

Thử nhanh hướng A bằng tay-code: tìm vị trí max của một đoạn bằng hai ô sparse table chồng nhau.

In [11]:
nums = [3, 2, 1, 6, 0, 5]
n = len(nums)
# table[0][i] = i ; table[k][i] = vị trí max trong nums[i .. i + 2^k - 1]
table = [list(range(n))]
k = 1
while (1 << k) <= n:
    prev = table[-1]
    row = []
    for i in range(n - (1 << k) + 1):
        a, b = prev[i], prev[i + (1 << (k - 1))]
        row.append(a if nums[a] > nums[b] else b)
    table.append(row)
    k += 1
print(table)

def query(l, r):                               # vị trí max trong nums[l..r]
    k = (r - l + 1).bit_length() - 1
    a, b = table[k][l], table[k][r - (1 << k) + 1]
    return a if nums[a] > nums[b] else b

print(query(0, 5), query(0, 2), query(4, 5), query(1, 2))   # 3, 0, 5, 1

[[0, 1, 2, 3, 4, 5], [0, 1, 3, 3, 5], [3, 3, 3]]
3 0 5 1


## 7. Các cách tiếp theo

### Cách 2 — Ngăn xếp đơn điệu: dựng cây trong một lượt (`O(n)`)

**Giải thích từng đoạn:**

1. `stack` giữ **gai phải** của cây đang dựng: các nút có giá trị **giảm dần** từ đáy lên đỉnh.
2. Với mỗi `x` trong `nums`: tạo `node = TreeNode(x)`, đặt `last = None`.
3. **Trong khi** đỉnh ngăn xếp có giá trị **nhỏ hơn** `x`: pop nó, nhớ vào `last` (nút pop **sau cùng** là nút **lớn nhất** trong số bị pop, vì pop theo thứ tự tăng dần).
4. `node.left = last` — toàn bộ phần bị pop trở thành cây con trái của `node` (nút `last` đã mang theo các nút pop trước đó như cây con phải của nó).
5. Nếu ngăn xếp còn phần tử: `stack[-1].right = node` (`node` thành con phải của nút lớn hơn nó ngay trên).
6. Đẩy `node` vào ngăn xếp.
7. Gốc cây là phần tử ở **đáy** ngăn xếp: `stack[0]`.

In [12]:
def build_stack(nums):
    stack = []
    for x in nums:
        node = TreeNode(x)
        last = None
        while stack and stack[-1].val < x:
            last = stack.pop()
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[0] if stack else None

for nums, expected in examples:
    got = tree_to_list(build_stack(nums))
    print(nums, "->", got)
    assert got == expected

[3, 2, 1, 6, 0, 5] -> [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> [3, None, 2, None, 1]
[1] -> [1]
[1, 2, 3] -> [3, 2, None, 1]


In [13]:
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        stack = []
        for x in nums:
            node = TreeNode(x)
            last = None
            while stack and stack[-1].val < x:
                last = stack.pop()
            node.left = last
            if stack:
                stack[-1].right = node
            stack.append(node)
        return stack[0] if stack else None

print([tree_to_list(Solution().constructMaximumBinaryTree(v)) for v, _ in examples])

[[6, 3, 5, None, 2, 0, None, None, 1], [3, None, 2, None, 1], [1], [3, 2, None, 1]]


**Độ phức tạp (cách 2):**

- **Time:** `O(n)` — mỗi phần tử được đẩy đúng một lần và pop tối đa một lần (phân tích khấu hao): vòng `while` tổng cộng chạy không quá `n` lần qua cả quá trình.
- **Space:** `O(n)` cho ngăn xếp (và cây trả về). Không đệ quy.

### Cách 3 — Đệ quy theo đoạn `[l, r]` + Sparse Table (`O(n log n)`)

Giữ nguyên cấu trúc đệ quy của cách 1 nhưng:

1. **Không cắt lát**: truyền `l, r`.
2. **Không quét tìm max**: dùng `query(l, r)` của bảng thưa → `O(1)`.

**Giải thích từng đoạn:**

1. Dựng `table` như mục 6 (`O(n log n)`).
2. `build(l, r)`: nếu `l > r` trả `None`; `i = query(l, r)`; `TreeNode(nums[i], build(l, i-1), build(i+1, r))`.
3. Gọi `build(0, n-1)`.

In [14]:
def build_sparse(nums):
    n = len(nums)
    if n == 0:
        return None
    table = [list(range(n))]
    k = 1
    while (1 << k) <= n:
        prev = table[-1]
        half = 1 << (k - 1)
        row = []
        for i in range(n - (1 << k) + 1):
            a, b = prev[i], prev[i + half]
            row.append(a if nums[a] > nums[b] else b)
        table.append(row)
        k += 1

    def query(l, r):
        k = (r - l + 1).bit_length() - 1
        a, b = table[k][l], table[k][r - (1 << k) + 1]
        return a if nums[a] > nums[b] else b

    def build(l, r):
        if l > r:
            return None
        i = query(l, r)
        return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))

    return build(0, n - 1)

for nums, expected in examples:
    got = tree_to_list(build_sparse(nums))
    print(nums, "->", got)
    assert got == expected

[3, 2, 1, 6, 0, 5] -> [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> [3, None, 2, None, 1]
[1] -> [1]
[1, 2, 3] -> [3, 2, None, 1]


In [15]:
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        n = len(nums)
        table = [list(range(n))]
        k = 1
        while (1 << k) <= n:
            prev, half = table[-1], 1 << (k - 1)
            row = []
            for i in range(n - (1 << k) + 1):
                a, b = prev[i], prev[i + half]
                row.append(a if nums[a] > nums[b] else b)
            table.append(row)
            k += 1

        def query(l, r):
            k = (r - l + 1).bit_length() - 1
            a, b = table[k][l], table[k][r - (1 << k) + 1]
            return a if nums[a] > nums[b] else b

        def build(l, r):
            if l > r:
                return None
            i = query(l, r)
            return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))

        return build(0, n - 1)

print([tree_to_list(Solution().constructMaximumBinaryTree(v)) for v, _ in examples])

[[6, 3, 5, None, 2, 0, None, None, 1], [3, None, 2, None, 1], [1], [3, 2, None, 1]]


**Độ phức tạp (cách 3):**

- **Time:** `O(n log n)` dựng bảng + `O(n)` dựng cây (mỗi nút một truy vấn `O(1)`).
- **Space:** `O(n log n)` cho bảng thưa, cộng ngăn xếp đệ quy `O(h)` (tới `n` với cây lệch).

Cách này không nhanh hơn cách 2 về lý thuyết nhưng **hữu ích khi bạn đã có sẵn cấu trúc tìm max theo đoạn** (hoặc khi đề đổi thành "tìm vị trí max của đoạn" nhiều lần).

### Cách 4 — Luật "cha": so hai phần tử lớn hơn gần nhất (`O(n)`)

**Ý tưởng:** với mỗi vị trí `i`, gọi

- `L[i]` = vị trí phần tử **lớn hơn gần nhất bên trái** (hoặc `-1` nếu không có),
- `R[i]` = vị trí phần tử **lớn hơn gần nhất bên phải** (hoặc `n` nếu không có).

Khi dựng theo đề, cha của nút `nums[i]` là **phần tử nhỏ hơn** trong hai phần tử `nums[L[i]]` và `nums[R[i]]` (những cái tồn tại). Lý do: cả hai đều "bao" `i` từ hai phía; cái **nhỏ hơn** sẽ bị chọn làm gốc của đoạn chứa `i` **sau** cái lớn hơn, nên nó là tổ tiên gần nhất. Nếu cha là `R[i]` thì `i` là **con trái** (`i` nằm bên trái cha); nếu cha là `L[i]` thì `i` là **con phải**. Không có cả hai → `i` là gốc.

**Giải thích từng đoạn:**

1. Tính `L` bằng ngăn xếp đơn điệu quét từ trái sang phải; tính `R` bằng quét từ phải sang trái (hoặc chính ngăn xếp đó khi pop).
2. Tạo sẵn `nodes = [TreeNode(x) for x in nums]`.
3. Với mỗi `i`: chọn cha theo luật trên và gắn con vào đúng phía.

In [16]:
def build_parent_rule(nums):
    n = len(nums)
    L = [-1] * n
    R = [n] * n
    st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] < x:
            R[st.pop()] = i                # i là "lớn hơn gần nhất bên phải" của phần tử bị pop
        L[i] = st[-1] if st else -1        # đỉnh còn lại là "lớn hơn gần nhất bên trái"
        st.append(i)

    nodes = [TreeNode(x) for x in nums]
    root = None
    for i in range(n):
        has_l, has_r = L[i] >= 0, R[i] < n
        if not has_l and not has_r:
            root = nodes[i]
        elif has_l and (not has_r or nums[L[i]] < nums[R[i]]):
            nodes[L[i]].right = nodes[i]   # cha là L[i], i nằm bên phải cha
        else:
            nodes[R[i]].left = nodes[i]    # cha là R[i], i nằm bên trái cha
    return root

for nums, expected in examples:
    got = tree_to_list(build_parent_rule(nums))
    print(nums, "->", got)
    assert got == expected

[3, 2, 1, 6, 0, 5] -> [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> [3, None, 2, None, 1]
[1] -> [1]
[1, 2, 3] -> [3, 2, None, 1]


In [17]:
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        n = len(nums)
        L, R, st = [-1] * n, [n] * n, []
        for i, x in enumerate(nums):
            while st and nums[st[-1]] < x:
                R[st.pop()] = i
            L[i] = st[-1] if st else -1
            st.append(i)

        nodes = [TreeNode(x) for x in nums]
        root = None
        for i in range(n):
            has_l, has_r = L[i] >= 0, R[i] < n
            if not has_l and not has_r:
                root = nodes[i]
            elif has_l and (not has_r or nums[L[i]] < nums[R[i]]):
                nodes[L[i]].right = nodes[i]
            else:
                nodes[R[i]].left = nodes[i]
        return root

print([tree_to_list(Solution().constructMaximumBinaryTree(v)) for v, _ in examples])

[[6, 3, 5, None, 2, 0, None, None, 1], [3, None, 2, None, 1], [1], [3, 2, None, 1]]


**Độ phức tạp (cách 4):**

- **Time:** `O(n)` — một lượt ngăn xếp để tính `L`, `R`; một lượt gắn cha.
- **Space:** `O(n)` cho `L`, `R`, `nodes`, ngăn xếp. Không đệ quy.

Quan trọng: cách 4 cho thấy **cấu trúc cây hoàn toàn được xác định bởi hai mảng "lớn hơn gần nhất"**, đây cũng là cốt lõi của các bài như *Largest Rectangle in Histogram* hay *Sum of Subarray Minimums*.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (ngăn xếp) trên `[3, 2, 1, 6, 0, 5]`

| Bước | `x` | Ngăn xếp trước | Pop (theo thứ tự) | `node.left` | `stack[-1].right = node`? | Ngăn xếp sau |
|---|---|---|---|---|---|---|
| 1 | 3 | `[]` | — | `None` | không (rỗng) | `[3]` |
| 2 | 2 | `[3]` | — | `None` | `3.right = 2` | `[3, 2]` |
| 3 | 1 | `[3, 2]` | — | `None` | `2.right = 1` | `[3, 2, 1]` |
| 4 | 6 | `[3, 2, 1]` | `1`, `2`, `3` | `3` (`last`) | không (rỗng) | `[6]` |
| 5 | 0 | `[6]` | — | `None` | `6.right = 0` | `[6, 0]` |
| 6 | 5 | `[6, 0]` | `0` | `0` | `6.right = 5` | `[6, 5]` |

Gốc = `stack[0]` = `6`. Ở bước 4, ba nút `1, 2, 3` bị pop: `last = 3` mang theo chuỗi `3 → 2 → 1` (đã nối từ các bước trước), trở thành cây con trái của `6`. Ở bước 6, `6.right` được **ghi đè** từ `0` sang `5`, và `0` đã thành con trái của `5`.

In [18]:
def trace_stack(nums):
    stack = []
    for step, x in enumerate(nums, 1):
        node = TreeNode(x)
        before = [s.val for s in stack]
        popped, last = [], None
        while stack and stack[-1].val < x:
            last = stack.pop()
            popped.append(last.val)
        node.left = last
        linked = None
        if stack:
            stack[-1].right = node
            linked = f"{stack[-1].val}.right = {x}"
        stack.append(node)
        print(f"bước {step}: x={x} | trước {before} | pop {popped} | left={last.val if last else None}"
              f" | {linked or 'không gắn (ngăn xếp rỗng)'} | sau {[s.val for s in stack]}")
    return stack[0]

root = trace_stack([3, 2, 1, 6, 0, 5])
print("kết quả:", tree_to_list(root))

bước 1: x=3 | trước [] | pop [] | left=None | không gắn (ngăn xếp rỗng) | sau [3]
bước 2: x=2 | trước [3] | pop [] | left=None | 3.right = 2 | sau [3, 2]
bước 3: x=1 | trước [3, 2] | pop [] | left=None | 2.right = 1 | sau [3, 2, 1]
bước 4: x=6 | trước [3, 2, 1] | pop [1, 2, 3] | left=3 | không gắn (ngăn xếp rỗng) | sau [6]
bước 5: x=0 | trước [6] | pop [] | left=None | 6.right = 0 | sau [6, 0]
bước 6: x=5 | trước [6, 0] | pop [0] | left=0 | 6.right = 5 | sau [6, 5]
kết quả: [6, 3, 5, None, 2, 0, None, None, 1]


### 8.2. Trace cách 1 (đệ quy) — cây gọi hàm

Hàm `trace_rec` in mỗi lần gọi theo mức thụt lề để thấy đoạn con được chia thế nào (dùng đoạn chỉ số cho gọn).

In [19]:
def trace_rec(nums):
    def build(l, r, depth, tag):
        pad = "    " * depth
        if l > r:
            print(f"{pad}{tag}: đoạn rỗng -> None")
            return None
        i = max(range(l, r + 1), key=lambda k: nums[k])
        print(f"{pad}{tag}: đoạn {nums[l:r + 1]} -> max {nums[i]} ở vị trí {i}; trái {nums[l:i]}, phải {nums[i + 1:r + 1]}")
        node = TreeNode(nums[i])
        node.left = build(l, i - 1, depth + 1, "trái")
        node.right = build(i + 1, r, depth + 1, "phải")
        return node
    return build(0, len(nums) - 1, 0, "gốc")

root = trace_rec([3, 2, 1, 6, 0, 5])
print(tree_to_list(root))

gốc: đoạn [3, 2, 1, 6, 0, 5] -> max 6 ở vị trí 3; trái [3, 2, 1], phải [0, 5]
    trái: đoạn [3, 2, 1] -> max 3 ở vị trí 0; trái [], phải [2, 1]
        trái: đoạn rỗng -> None
        phải: đoạn [2, 1] -> max 2 ở vị trí 1; trái [], phải [1]
            trái: đoạn rỗng -> None
            phải: đoạn [1] -> max 1 ở vị trí 2; trái [], phải []
                trái: đoạn rỗng -> None
                phải: đoạn rỗng -> None
    phải: đoạn [0, 5] -> max 5 ở vị trí 5; trái [0], phải []
        trái: đoạn [0] -> max 0 ở vị trí 4; trái [], phải []
            trái: đoạn rỗng -> None
            phải: đoạn rỗng -> None
        phải: đoạn rỗng -> None
[6, 3, 5, None, 2, 0, None, None, 1]


### 8.3. Trace cách 4 — bảng `L`, `R` và cha

Với `nums = [3, 2, 1, 6, 0, 5]`:

| `i` | `nums[i]` | `L[i]` | `R[i]` | cha | phía |
|---|---|---|---|---|---|
| 0 | 3 | −1 | 3 (giá trị 6) | `6` (R) | con **trái** của 6 |
| 1 | 2 | 0 (giá trị 3) | 3 (giá trị 6) | `3` (nhỏ hơn) | con **phải** của 3 |
| 2 | 1 | 1 (giá trị 2) | 3 (giá trị 6) | `2` | con **phải** của 2 |
| 3 | 6 | −1 | 6 (không có) | — | **gốc** |
| 4 | 0 | 3 (giá trị 6) | 5 (giá trị 5) | `5` (nhỏ hơn) | con **trái** của 5 |
| 5 | 5 | 3 (giá trị 6) | 6 (không có) | `6` (L) | con **phải** của 6 |

In [20]:
def trace_parent_rule(nums):
    n = len(nums)
    L, R, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] < x:
            R[st.pop()] = i
        L[i] = st[-1] if st else -1
        st.append(i)
    print("L =", L)
    print("R =", R)
    for i in range(n):
        lv = nums[L[i]] if L[i] >= 0 else None
        rv = nums[R[i]] if R[i] < n else None
        if lv is None and rv is None:
            print(f"i={i} x={nums[i]}: L=None R=None -> GỐC")
        elif lv is not None and (rv is None or lv < rv):
            print(f"i={i} x={nums[i]}: L={lv} R={rv} -> cha {lv} (nhỏ hơn), x là con PHẢI")
        else:
            print(f"i={i} x={nums[i]}: L={lv} R={rv} -> cha {rv} (nhỏ hơn), x là con TRÁI")

trace_parent_rule([3, 2, 1, 6, 0, 5])

L = [-1, 0, 1, -1, 3, 3]
R = [3, 3, 3, 6, 5, 6]
i=0 x=3: L=None R=6 -> cha 6 (nhỏ hơn), x là con TRÁI
i=1 x=2: L=3 R=6 -> cha 3 (nhỏ hơn), x là con PHẢI
i=2 x=1: L=2 R=6 -> cha 2 (nhỏ hơn), x là con PHẢI
i=3 x=6: L=None R=None -> GỐC
i=4 x=0: L=6 R=5 -> cha 5 (nhỏ hơn), x là con TRÁI
i=5 x=5: L=6 R=None -> cha 6 (nhỏ hơn), x là con PHẢI


## 9. Code nộp LeetCode

Đây là cách 2 (ngăn xếp đơn điệu) — ngắn, `O(n)`, **không đệ quy**. Khi nộp lên LeetCode không cần định nghĩa `TreeNode` và `Optional`, `List` đã được import sẵn; ở đây ta giữ lại ở đầu notebook để chạy cục bộ.

In [21]:
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        stack = []
        for x in nums:
            node = TreeNode(x)
            last = None
            while stack and stack[-1].val < x:
                last = stack.pop()
            node.left = last
            if stack:
                stack[-1].right = node
            stack.append(node)
        return stack[0] if stack else None


sol = Solution()
for nums, expected in examples:
    got = tree_to_list(sol.constructMaximumBinaryTree(nums))
    print(nums, "->", got)
    assert got == expected
    assert is_valid_max_tree(sol.constructMaximumBinaryTree(nums), nums)
print("Chạy ví dụ: OK")

[3, 2, 1, 6, 0, 5] -> [6, 3, 5, None, 2, 0, None, None, 1]
[3, 2, 1] -> [3, None, 2, None, 1]
[1] -> [1]
[1, 2, 3] -> [3, 2, None, 1]
Chạy ví dụ: OK


## 10. Vì sao đúng

**Phát biểu bất biến (cách 2).** Sau khi xử lý `nums[0..i]`, cây mà ngăn xếp đại diện (gốc là `stack[0]`, cộng toàn bộ cây con treo trên các nút) **chính là** cây lớn nhất dựng từ `nums[0..i]`, và ngăn xếp chứa đúng **gai phải** của cây đó, với giá trị giảm dần từ đáy lên đỉnh.

**Chứng minh quy nạp.**

- Cơ sở: `i = 0` — một nút, ngăn xếp `[nums[0]]`.
- Bước: thêm `x = nums[i]`. Theo định nghĩa, `x` ở cuối mảng, nên trong cây mới `x` nằm trên **gai phải**. Các nút trên gai phải cũ nhỏ hơn `x` (đang ở đỉnh ngăn xếp) sẽ nằm **dưới** `x` (vì chúng bị `x` lấn át khi `x` là max của đoạn chứa chúng cùng `x`), và vì chúng nằm **bên trái** `x` trong mảng nên thành cây con **trái** của `x`; đó chính là phần bị pop (`node.left = last`, `last` là nút lớn nhất trong số bị pop và đã mang theo các nút nhỏ hơn dưới dạng cây con phải). Nút còn lại ở đỉnh ngăn xếp **lớn hơn** `x` và nằm bên trái `x`; `x` thuộc đoạn bên phải của nó nên là con phải của nó (`stack[-1].right = node`, ghi đè con phải cũ, vốn đã chuyển xuống làm con trái của `x`). Gai phải mới là phần ngăn xếp còn lại cộng `x`: vẫn giảm dần.

**Tính duy nhất và kiểm chứng.** Với giá trị khác nhau, cây thoả "duyệt giữa = mảng" và "cha lớn hơn con" là duy nhất. Ta kiểm chứng bằng code: cả bốn cách trả về cây thoả hai tính chất, và các cây trùng nhau từng nút.

In [22]:
def same_tree(a, b):
    if a is None or b is None:
        return a is b
    return a.val == b.val and same_tree(a.left, b.left) and same_tree(a.right, b.right)

rng = random.Random(654)
count = 0
for _ in range(300):
    n = rng.randint(1, 12)
    nums = rng.sample(range(0, 30), n)
    trees = [build_slice(nums), build_stack(nums), build_sparse(nums), build_parent_rule(nums)]
    for t in trees:
        assert is_valid_max_tree(t, nums), nums
    for t in trees[1:]:
        assert same_tree(trees[0], t), nums
    count += 1
print("đã kiểm chứng tính chất duyệt-giữa + heap và cấu trúc trùng nhau trên", count, "mảng ngẫu nhiên")

đã kiểm chứng tính chất duyệt-giữa + heap và cấu trúc trùng nhau trên 300 mảng ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy max + cắt lát | `O(n²)` xấu nhất | `O(n)` cây + bản sao lát | chuẩn đề bài, dùng làm oracle |
| 2. Ngăn xếp đơn điệu | `O(n)` | `O(n)` | **cách nộp chuẩn**, không đệ quy |
| 3. Sparse Table + đệ quy theo đoạn | `O(n log n)` | `O(n log n)` | khi cần truy vấn max theo đoạn nhiều lần |
| 4. Luật cha `L`, `R` | `O(n)` | `O(n)` | cho thấy cấu trúc cây chỉ phụ thuộc "lớn hơn gần nhất" |

Cách 2: mỗi phần tử vào ngăn xếp một lần và ra tối đa một lần, nên tổng số lần lặp của `while` qua toàn bộ vòng `for` không vượt quá `n` → `O(n)` khấu hao. Với `n ≤ 1000` cả bốn cách đều chạy rất nhanh; sự khác biệt thể hiện rõ khi `n` lớn hơn nhiều.

## 12. Bộ test

Gồm: ví dụ đề (so sánh **list kiểu LeetCode** đúng như đề), edge case (một phần tử, hai phần tử, tăng dần, giảm dần, có số `0`, zigzag), đối chiếu bốn cách với nhau và với **hàm kiểm tra tính hợp lệ** (`is_valid_max_tree`: duyệt giữa = mảng và tính chất heap), mảng ngẫu nhiên có seed đúng ràng buộc, và test lớn nhất `n = 1000`.

In [23]:
METHODS = {
    "slice": build_slice,
    "stack": build_stack,
    "sparse": build_sparse,
    "parent": build_parent_rule,
}

def check_all(nums, expected=None):
    ref = None
    for name, fn in METHODS.items():
        root = fn(list(nums))
        assert is_valid_max_tree(root, nums), (name, nums)
        lst = tree_to_list(root)
        if expected is not None:
            assert lst == expected, (name, nums, lst, expected)
        if ref is None:
            ref = lst
        assert lst == ref, (name, nums)

for nums, expected in examples:
    check_all(nums, expected)

# edge case
check_all([0], [0])
check_all([5, 7], [7, 5])
check_all([7, 5], [7, None, 5])
check_all([1, 2, 3, 4], [4, 3, None, 2, None, 1])             # tăng dần: lệch trái
check_all([4, 3, 2, 1], [4, None, 3, None, 2, None, 1])       # giảm dần: lệch phải
check_all([2, 0, 3, 1], [3, 2, 1, None, 0])                   # có số 0
check_all([1, 3, 2], [3, 1, 2])                               # max ở giữa
check_all([2, 4, 1, 5, 3], [5, 4, 3, 2, 1])                   # zigzag
check_all([0, 1000, 500], [1000, 0, 500])                     # biên giá trị
print("edge case: OK")

edge case: OK


In [24]:
# Đối chiếu nhiều mảng ngẫu nhiên nhỏ (hoán vị và tập con khác nhau), seed cố định
rng = random.Random(2025)
count = 0
for _ in range(1500):
    n = rng.randint(1, 15)
    nums = rng.sample(range(0, 40), n)
    check_all(nums)
    count += 1
print("đã đối chiếu", count, "mảng ngẫu nhiên nhỏ: OK")

# n lớn: giá trị rút không lặp từ [0, 1000]
fast = {k: METHODS[k] for k in ("stack", "sparse", "parent")}
for _ in range(60):
    n = rng.randint(100, 1000)
    nums = rng.sample(range(0, 1001), n)
    ref = tree_to_list(build_slice(nums))
    for name, fn in fast.items():
        assert tree_to_list(fn(list(nums))) == ref, (name, n)
print("n 100..1000: OK")

đã đối chiếu 1500 mảng ngẫu nhiên nhỏ: OK


n 100..1000: OK


In [25]:
# Test lớn nhất theo ràng buộc: n = 1000, các dạng "xấu" cho cách 1
shapes = {
    "tăng dần": list(range(0, 1000)),
    "giảm dần": list(range(999, -1, -1)),
    "ngẫu nhiên": random.Random(1).sample(range(0, 1001), 1000),
    "zigzag": [x for pair in zip(range(500, 1000), range(0, 500)) for x in pair],
}
for name, nums in shapes.items():
    row = []
    ref = None
    for mname, fn in METHODS.items():
        t0 = time.perf_counter()
        root = fn(list(nums))
        dt = (time.perf_counter() - t0) * 1000
        assert is_valid_max_tree(root, nums), (name, mname)
        lst = tree_to_list(root)
        if ref is None:
            ref = lst
        assert lst == ref, (name, mname)
        row.append(f"{mname} {dt:6.2f} ms")
    print(f"{name:11}", " | ".join(row))

tăng dần    slice  30.41 ms | stack   0.40 ms | sparse   1.63 ms | parent   0.58 ms
giảm dần    slice  11.52 ms | stack   0.35 ms | sparse   1.75 ms | parent   0.49 ms
ngẫu nhiên  slice   1.13 ms | stack   0.51 ms | sparse   1.55 ms | parent   0.63 ms
zigzag      slice   8.72 ms | stack   0.38 ms | sparse   1.52 ms | parent   0.53 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên cộng độ lệch chỉ số khi tìm max trên lát cắt

**Code sai** (đệ quy theo đoạn `[l, r]` nhưng tìm vị trí trên lát cắt):

```python
sub = nums[l:r + 1]
i = sub.index(max(sub))            # i là vị trí TRONG LÁT CẮT, chưa cộng l
node = TreeNode(nums[i], build(l, i - 1), build(i + 1, r))
```

**Vì sao sai:** `i` tính từ đầu lát cắt, còn `build(l, i - 1)` và `nums[i]` cần chỉ số **toàn cục**. Khi `l > 0`, `nums[i]` lấy sai phần tử, đoạn con bị chia sai. Phải dùng `i = l + sub.index(...)`.

In [26]:
def wrong_offset(nums):
    def build(l, r):
        if l > r:
            return None
        sub = nums[l:r + 1]
        i = sub.index(max(sub))                    # SAI: thiếu + l
        return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))
    return build(0, len(nums) - 1)

def right_offset(nums):
    def build(l, r):
        if l > r:
            return None
        sub = nums[l:r + 1]
        i = l + sub.index(max(sub))                # ĐÚNG
        return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))
    return build(0, len(nums) - 1)

nums = [3, 2, 1, 6, 0, 5]
try:
    print("sai :", tree_to_list(wrong_offset(nums)))
except RecursionError:
    print("sai : RecursionError (đệ quy không bao giờ dừng)")
print("đúng:", tree_to_list(right_offset(nums)))

sai : RecursionError (đệ quy không bao giờ dừng)
đúng: [6, 3, 5, None, 2, 0, None, None, 1]


### Lỗi 2: Cắt lát sai làm đệ quy không dừng

**Code sai:**

```python
return TreeNode(m, build(nums[:i]), build(nums[i:]))       # nums[i:] vẫn chứa phần tử lớn nhất!
```

**Vì sao sai:** phần bên phải phải bắt đầu **sau** vị trí `i`, tức `nums[i+1:]`. Nếu dùng `nums[i:]` thì phần tử lớn nhất xuất hiện lại trong đoạn con (độ dài không giảm khi `i = 0`) và đệ quy **lặp vô hạn** (`RecursionError`).

In [27]:
def wrong_slice(nums):
    if not nums:
        return None
    m = max(nums)
    i = nums.index(m)
    return TreeNode(m, wrong_slice(nums[:i]), wrong_slice(nums[i:]))      # SAI: nums[i:]

try:
    wrong_slice([3, 2, 1])
except RecursionError:
    print("sai : RecursionError")
print("đúng:", tree_to_list(build_slice([3, 2, 1])))

sai : RecursionError
đúng: [3, None, 2, None, 1]


### Lỗi 3: Quên gắn lại phần bị pop làm cây con trái (`node.left = last`)

**Code sai:**

```python
while stack and stack[-1].val < x:
    stack.pop()                    # bỏ đi, không giữ lại
# thiếu: node.left = last
```

**Vì sao sai:** những nút bị pop không "biến mất" — chúng phải trở thành **cây con trái** của `node`. Nếu bỏ đi, cả một nhánh bị mất khỏi cây.

In [28]:
def wrong_lose_popped(nums):
    stack = []
    for x in nums:
        node = TreeNode(x)
        while stack and stack[-1].val < x:
            stack.pop()                          # SAI: mất cây con
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[0] if stack else None

nums = [3, 2, 1, 6, 0, 5]
print("sai :", tree_to_list(wrong_lose_popped(nums)))
print("đúng:", tree_to_list(build_stack(nums)))

sai : [6, None, 5]
đúng: [6, 3, 5, None, 2, 0, None, None, 1]


### Lỗi 4: Trả về `stack[-1]` thay vì `stack[0]`

**Code sai:**

```python
return stack[-1]       # đỉnh ngăn xếp = nút nhỏ nhất trên gai phải
```

**Vì sao sai:** ngăn xếp giảm dần từ **đáy lên đỉnh**: đáy (`stack[0]`) là nút lớn nhất = **gốc**, đỉnh (`stack[-1]`) là nút cuối của gai phải. Trả `stack[-1]` chỉ trả một cây con.

In [29]:
def wrong_return_top(nums):
    stack = []
    for x in nums:
        node = TreeNode(x)
        last = None
        while stack and stack[-1].val < x:
            last = stack.pop()
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[-1]                               # SAI

nums = [3, 2, 1, 6, 0, 5]
print("sai :", tree_to_list(wrong_return_top(nums)))
print("đúng:", tree_to_list(build_stack(nums)))

sai : [5, 0]
đúng: [6, 3, 5, None, 2, 0, None, None, 1]


### Lỗi 5: Không xử lý mảng rỗng

**Code sai:**

```python
def build(nums):
    m = max(nums)                 # ValueError khi nums rỗng
    ...
```

**Vì sao sai:** đệ quy luôn tạo ra các đoạn **rỗng** (khi max nằm ở đầu hoặc cuối đoạn). `max([])` gây `ValueError: max() arg is an empty sequence`. Trường hợp cơ sở `if not nums: return None` là bắt buộc.

In [30]:
def wrong_no_base(nums):
    m = max(nums)                                  # SAI: không có điều kiện dừng
    i = nums.index(m)
    return TreeNode(m, wrong_no_base(nums[:i]), wrong_no_base(nums[i + 1:]))

try:
    wrong_no_base([3, 2, 1])
except ValueError as e:
    print("ValueError:", e)

ValueError: max() iterable argument is empty


## 14. Mẹo & nhận dạng pattern

- **"Gốc là max của đoạn, chia đôi tại đó" = Cartesian tree.** Nhận ra tên là đủ để nhớ mọi tính chất: in-order trả lại mảng, cha lớn hơn con, dựng trong `O(n)` bằng ngăn xếp.
- **Ngăn xếp đơn điệu** xuất hiện khi câu hỏi là "phần tử lớn/nhỏ hơn **gần nhất** bên trái/phải" (Next Greater Element, Daily Temperatures, Largest Rectangle in Histogram, Sum of Subarray Minimums). Bài này là phiên bản "dựng cả cây" của cùng ý tưởng.
- **Đệ quy "chia đôi tại max/min"** gặp nhiều trong: Largest Rectangle (chia tại cột thấp nhất), Burst Balloons (chia tại quả nổ cuối), RMQ. Hãy tự hỏi: "có thể tìm vị trí cực trị của đoạn nhanh hơn quét không?" → sparse table / segment tree.
- **Kiểm thử tốt:** vì đáp án là duy nhất, nhưng ta vẫn nên viết hàm kiểm tra **tính chất** (in-order + heap) độc lập với cách dựng, rồi so cấu trúc với oracle đệ quy.
- **Cây lệch sâu `n`:** nhớ giới hạn đệ quy; ưu tiên các cách dùng ngăn xếp.
- **Quy ước `None`:** `tree_to_list` bỏ `None` thừa ở cuối; khi so sánh với đề nhớ chuẩn hoá về cùng dạng (đề cũng bỏ `null` thừa cuối).

## 15. Biến thể

1. **LeetCode 998 — Maximum Binary Tree II:** cho cây lớn nhất `root` dựng từ `a` và một giá trị `val`; trả cây dựng từ `a + [val]` (thêm vào **cuối**). Chỉ cần đi theo gai phải.
2. **Cây nhỏ nhất (Min Cartesian tree):** gốc là phần tử **nhỏ nhất**; đổi dấu so sánh.
3. **Có phần tử trùng nhau:** phải quy ước chọn max đầu tiên hay cuối cùng (ví dụ cách 2 với `<` hay `<=`); đề này cấm trùng để tránh nhập nhằng.
4. **Largest Rectangle in Histogram (84):** dùng đúng ngăn xếp đơn điệu này, hoặc dựng cây Descartes (min) rồi tính `chiều cao × độ rộng` của mỗi nút.
5. **Sum of Subarray Minimums (907), Sum of Subarray Ranges (2104):** dùng mảng `L`, `R` của cách 4 để đếm số đoạn mà mỗi phần tử là cực trị.
6. **Treap (cây tìm kiếm nhị phân + heap):** cấu trúc dữ liệu cân bằng ngẫu nhiên chính là cây Descartes của (khoá, độ ưu tiên ngẫu nhiên).

Dưới đây là biến thể 1 (998), kiểm tra bằng cách so với dựng lại từ đầu:

In [31]:
def insert_into_max_tree(root, val):
    if root is None or val > root.val:
        return TreeNode(val, root)             # root cũ thành con trái của nút mới
    root.right = insert_into_max_tree(root.right, val)
    return root

rng = random.Random(998)
for _ in range(500):
    n = rng.randint(1, 10)
    pool = rng.sample(range(0, 40), n + 1)
    nums, val = pool[:n], pool[n]
    got = insert_into_max_tree(build_stack(nums), val)
    assert same_tree(got, build_stack(nums + [val])), (nums, val)
print("998 OK:", tree_to_list(insert_into_max_tree(build_stack([3, 2, 1, 6, 0]), 5)))

998 OK: [6, 3, 5, None, 2, 0, None, None, 1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Max + cắt lát | Dịch nguyên văn: tìm max, cắt hai nửa, đệ quy | `O(n²)` xấu nhất | `O(n)` + bản sao | Oracle; `n` nhỏ (≤ 1000 vẫn đủ nhanh) |
| 2. Ngăn xếp đơn điệu | Giữ gai phải; pop phần nhỏ hơn làm con trái | `O(n)` | `O(n)` | **Cách nộp chuẩn**, không đệ quy |
| 3. Sparse Table + đoạn | Tìm vị trí max đoạn trong `O(1)` rồi đệ quy | `O(n log n)` | `O(n log n)` | Khi cần truy vấn max theo đoạn nhiều lần |
| 4. Luật cha (`L`, `R`) | Cha = nhỏ hơn trong hai "lớn hơn gần nhất" | `O(n)` | `O(n)` | Cần hiểu cấu trúc; chuyển sang các bài đếm đoạn |

Kết luận: nộp **cách 2**. Cách 1 giữ lại làm công cụ kiểm thử. Cách 4 giúp mở rộng sang các bài dùng `L`, `R`.

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh.

In [32]:
nums = [4, 9, 2, 7]
print(max(nums), nums.index(max(nums)))
print(nums[:1], nums[2:])
print(tree_to_list(build_stack([4, 9, 2, 7])))
print(tree_to_list(build_stack([1, 2, 3])))
print(inorder(build_stack([4, 9, 2, 7])))

9 1
[4] [2, 7]
[9, 4, 7, None, None, 2]
[3, 2, None, 1]
[4, 9, 2, 7]


### Bài 2 (tính tay)

Dựng tay cây lớn nhất từ `nums = [5, 2, 7, 1, 6]`. Gán cây dưới dạng list kiểu LeetCode vào `my_list` (nhớ bỏ `None` thừa ở cuối). Hiện để `None`.

In [33]:
my_list = None
# Sau khi điền, bỏ comment để kiểm tra:
# assert my_list == tree_to_list(build_slice([5, 2, 7, 1, 6]))

### Bài 3 (điền TODO)

Hoàn thành vòng lặp của ngăn xếp đơn điệu bên dưới (`TODO` ở bốn chỗ).

In [34]:
def build_stack_todo(nums):
    stack = []
    for x in nums:
        node = TreeNode(x)
        last = None
        # TODO 1: trong khi đỉnh ngăn xếp nhỏ hơn x thì pop và lưu vào last
        # TODO 2: gắn node.left = last
        # TODO 3: nếu ngăn xếp còn phần tử thì gắn stack[-1].right = node
        # TODO 4: đẩy node vào ngăn xếp
        pass
    # TODO 5: trả về gốc (nhớ đáy hay đỉnh?)
    return None

# Kiểm tra (bỏ comment sau khi viết xong):
# for nums, expected in examples:
#     assert tree_to_list(build_stack_todo(nums)) == expected

### Bài 4 (tự viết lại)

Viết `build_by_index(nums)`: đệ quy theo đoạn `[l, r]` (**không cắt lát**), tìm vị trí max bằng vòng `for` hoặc `max(range(...), key=...)`. Không xem lại code ở trên.

In [35]:
def build_by_index(nums):
    # TODO
    return None

# for nums, expected in examples:
#     assert tree_to_list(build_by_index(nums)) == expected
# assert is_valid_max_tree(build_by_index([2, 4, 1, 5, 3]), [2, 4, 1, 5, 3])

### Bài 5 (biến thể / mini challenge)

Viết `nearest_greater(nums)` trả về cặp `(L, R)` như cách 4 (`L[i]` = vị trí lớn hơn gần nhất bên trái hoặc `-1`; `R[i]` = bên phải hoặc `n`), rồi dùng nó để viết `parent_values(nums)` trả về list `p` với `p[i]` là **giá trị của cha** của `nums[i]` trong cây lớn nhất (hoặc `None` nếu là gốc). Kiểm tra bằng cây dựng từ cách 2.

In [36]:
def nearest_greater(nums):
    # TODO
    return None

def parent_values(nums):
    # TODO
    return None

def parent_values_oracle(nums):
    root = build_stack(nums)
    par = {}
    def walk(node):
        for ch in (node.left, node.right):
            if ch is not None:
                par[ch.val] = node.val
                walk(ch)
    walk(root)
    return [par.get(x) for x in nums]

# for nums in ([3, 2, 1, 6, 0, 5], [1, 2, 3], [4, 3, 2, 1], [2, 4, 1, 5, 3]):
#     assert parent_values(nums) == parent_values_oracle(nums), nums

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `max(nums)` là `9`, nằm ở vị trí `1`; `nums[:1]` là đoạn trước vị trí 1; `nums[2:]` từ vị trí 2 trở đi.
- Mức 2: `[4,9,2,7]`: gốc `9`; trái `[4]`, phải `[2,7]`; trong `[2,7]` gốc `7` với con trái `2`.
- Mức 3: kết quả theo thứ tự: `9 1`, `[4] [2, 7]`, `[9, 4, 7, None, None, 2]`, `[3, 2, None, 1]`, `[4, 9, 2, 7]`.

**Bài 2.**
- Mức 1: max là `7` ở vị trí 2; trái `[5, 2]`, phải `[1, 6]`.
- Mức 2: `[5, 2]` → gốc `5`, con phải `2`; `[1, 6]` → gốc `6`, con trái `1`.
- Mức 3: cây có gốc `7`, hai con `5` và `6`; `5` có con phải `2`, `6` có con trái `1`. Đọc từng tầng: `[7, 5, 6, None, 2, 1]`.

**Bài 3.**
- Mức 1: dùng `while stack and stack[-1].val < x:`.
- Mức 2: bên trong `while` có `last = stack.pop()`; sau `while`: `node.left = last`.
- Mức 3: `if stack: stack[-1].right = node`; `stack.append(node)`; cuối hàm `return stack[0] if stack else None`.

**Bài 4.**
- Mức 1: viết hàm con `build(l, r)`, điều kiện dừng `l > r` trả `None`.
- Mức 2: `i = max(range(l, r + 1), key=lambda k: nums[k])` đã là chỉ số toàn cục.
- Mức 3: `return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))`.

**Bài 5.**
- Mức 1: tính `L`, `R` bằng ngăn xếp như cách 4.
- Mức 2: với mỗi `i`, ứng viên cha là `nums[L[i]]` (nếu `L[i] >= 0`) và `nums[R[i]]` (nếu `R[i] < n`).
- Mức 3: cha là ứng viên **nhỏ hơn**; không có ứng viên nào thì `None`.

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
nums = [4, 9, 2, 7]
assert (max(nums), nums.index(max(nums))) == (9, 1)
assert (nums[:1], nums[2:]) == ([4], [2, 7])
assert tree_to_list(build_stack([4, 9, 2, 7])) == [9, 4, 7, None, None, 2]
assert tree_to_list(build_stack([1, 2, 3])) == [3, 2, None, 1]
assert inorder(build_stack([4, 9, 2, 7])) == [4, 9, 2, 7]
print("Bài 1: '9 1', '[4] [2, 7]', [9, 4, 7, None, None, 2], [3, 2, None, 1], [4, 9, 2, 7]")

# Bài 2
my_list = [7, 5, 6, None, 2, 1]
assert my_list == tree_to_list(build_slice([5, 2, 7, 1, 6]))
print("Bài 2: OK", my_list)

Bài 1: '9 1', '[4] [2, 7]', [9, 4, 7, None, None, 2], [3, 2, None, 1], [4, 9, 2, 7]
Bài 2: OK [7, 5, 6, None, 2, 1]


In [38]:
# Bài 3
def build_stack_todo(nums):
    stack = []
    for x in nums:
        node = TreeNode(x)
        last = None
        while stack and stack[-1].val < x:
            last = stack.pop()
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[0] if stack else None

for nums, expected in examples:
    assert tree_to_list(build_stack_todo(nums)) == expected
print("Bài 3: OK")

# Bài 4
def build_by_index(nums):
    def build(l, r):
        if l > r:
            return None
        i = max(range(l, r + 1), key=lambda k: nums[k])
        return TreeNode(nums[i], build(l, i - 1), build(i + 1, r))
    return build(0, len(nums) - 1)

for nums, expected in examples:
    assert tree_to_list(build_by_index(nums)) == expected
assert is_valid_max_tree(build_by_index([2, 4, 1, 5, 3]), [2, 4, 1, 5, 3])
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [39]:
# Bài 5
def nearest_greater(nums):
    n = len(nums)
    L, R, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] < x:
            R[st.pop()] = i
        L[i] = st[-1] if st else -1
        st.append(i)
    return L, R

def parent_values(nums):
    n = len(nums)
    L, R = nearest_greater(nums)
    res = []
    for i in range(n):
        cands = []
        if L[i] >= 0:
            cands.append(nums[L[i]])
        if R[i] < n:
            cands.append(nums[R[i]])
        res.append(min(cands) if cands else None)
    return res

def parent_values_oracle(nums):
    root = build_stack(nums)
    par = {}
    def walk(node):
        for ch in (node.left, node.right):
            if ch is not None:
                par[ch.val] = node.val
                walk(ch)
    walk(root)
    return [par.get(x) for x in nums]

for nums in ([3, 2, 1, 6, 0, 5], [1, 2, 3], [4, 3, 2, 1], [2, 4, 1, 5, 3]):
    assert parent_values(nums) == parent_values_oracle(nums), nums
rng = random.Random(5)
for _ in range(300):
    nums = rng.sample(range(0, 50), rng.randint(1, 12))
    assert parent_values(nums) == parent_values_oracle(nums), nums
print("Bài 5: OK", parent_values([3, 2, 1, 6, 0, 5]))

Bài 5: OK [6, 3, 2, None, 5, 6]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template dựng Cartesian tree bằng ngăn xếp đơn điệu:**

```python
class Solution:
    def constructMaximumBinaryTree(self, nums: List[int]) -> Optional[TreeNode]:
        stack = []                                   # giá trị giảm dần từ đáy lên đỉnh
        for x in nums:
            node = TreeNode(x)
            last = None
            while stack and stack[-1].val < x:       # pop mọi nút nhỏ hơn x
                last = stack.pop()
            node.left = last                         # phần bị pop thành con trái
            if stack:
                stack[-1].right = node               # x thành con phải của nút lớn hơn
            stack.append(node)
        return stack[0] if stack else None           # gốc ở ĐÁY
```

**Ghi nhớ một câu:** *"Pop những ai nhỏ hơn mình, người pop sau cùng làm con trái; người còn lại trên đỉnh nhận mình làm con phải; gốc ở đáy."*

**Checklist trước khi nộp:**

- [ ] `while` dùng `<` (giá trị khác nhau nên `<` và `<=` như nhau, nhưng nhất quán).
- [ ] `node.left = last` đặt **sau** vòng `while` (quên là mất cây con).
- [ ] Gắn `stack[-1].right = node` **trước** khi `append(node)`.
- [ ] Trả `stack[0]` (đáy), xử lý `nums` rỗng bằng `None` nếu cần.
- [ ] Nếu dùng đệ quy theo đoạn: điều kiện dừng `l > r`, chỉ số max là **toàn cục** (nhớ `+ l` khi tìm trên lát cắt), nới `setrecursionlimit` cho cây lệch 1000 tầng.
- [ ] Đã thử: `[1]`, tăng dần, giảm dần, max ở giữa, có số `0`, `n = 1000`.
- [ ] Đã kiểm tra bằng hàm tính chất: duyệt giữa = mảng, cha lớn hơn con.